# Cleaning Data

Before making changes to a dataset, you should first understand what each variable represents. Ideally, the data will include a README file or data dictionary explaining the variables, their measurement scales, and any coding conventions.

If the data come from your own experiment, you will hopefully already know this information. In some cases, however, you may only have the dataset itself and need to make informed assumptions.

For this example, assume we only know that the data come from a Stroop experiment. We might reasonably interpret the variables as follows:

| Variable | Likely meaning | Appropriate type |
|---|---|---|
| `P` | Unique participant identifier | Character |
| `age` | Participant age in years | Numeric |
| `CongruentReactionTime` | Reaction time, in seconds, for congruent trials | Numeric |
| `Incongruent.RT` | Reaction time, in seconds, for incongruent trials | Numeric |
| `Edu` | Highest education level achieved | Factor |
| `Completed` | Whether the participant completed the experiment | Logical |

Once you understand the variables, you can begin cleaning the data.

### Renaming Columns

The original variable names are inconsistent: some are abbreviated, some are descriptive, and they use different naming conventions. We can standardise them with `names()`:

```r
names(messyDat) <- c(
  "pcp_id",
  "pcp_age",
  "congruent_rt",
  "incongruent_rt",
  "pcp_edu_level",
  "completed_exp"
)
```

The `names()` function can display the names of variables in an object or assign a new vector of names.

Check the updated variable names:

```r
names(messyDat)
```

```text
[1] "pcp_id"         "pcp_age"        "congruent_rt"
[4] "incongruent_rt" "pcp_edu_level"  "completed_exp"
```

### Identifying Unique Values

The `pcp_age` variable is currently stored as a character variable, but ages should be numeric. The `unique()` function can help identify unusual values:

```r
unique(messyDat$pcp_age)
```

```text
[1] "18"       "19"       ""         "20"       "22"       "-20"      "eighteen" "21"
```

`unique()` returns each distinct value only once. For example, even if `"19"` occurred 100 times, it would appear only once in this output.

This reveals several problems:

- `"eighteen"` has been entered as text rather than as the number `18`.
- `"-20"` is not a plausible age.
- `""` represents a missing value.

Because a column in a data frame must contain one consistent data type, the presence of text causes R to import the entire `pcp_age` column as character data.

### Replacing Values

First, replace `"eighteen"` with `18`:

```r
messyDat$pcp_age[messyDat$pcp_age == "eighteen"] <- 18
```

In plain language: within `pcp_age`, find rows where the value is `"eighteen"` and replace it with `18`.

However, the variable is still character data:

```r
class(messyDat$pcp_age)
```

```text
[1] "character"
```

Although `18` was entered as a number, R coerces it to character because the rest of the column is still character data.

Let's also take a look at the `pcp_edu_level` unique values:

```r
unique(messyDat$pcp_edu_level)
```
```text
[1] "School" "UG"     "PG"     "pg"   
```

The only issue appears to be that both `"pg"` and `"PG"` are used


### Changing a Variable Type

Use `as.numeric()` to convert the variable from character to numeric:

```r
messyDat$pcp_age <- as.numeric(messyDat$pcp_age)

print(messyDat$pcp_age)
class(messyDat$pcp_age)
```

```text
[1]  18  19  NA  20  22  22 -20  18  21
[1] "numeric"
```

The empty string, `""`, is converted to `NA`, which R uses to represent a missing value.

The remaining issue is the value `-20`. What to do depends on your knowledge of the study and the data-entry process. If there is evidence that this was intended to be `20`, replace it:

```r
messyDat$pcp_age[messyDat$pcp_age == -20] <- 20
```

If the intended value is uncertain, it may be more appropriate to replace `-20` with `NA` instead:

```r
messyDat$pcp_age[messyDat$pcp_age == -20] <- NA
```
Finally, we can fix our `pcp_edu_level` variable in the same way:

```r
messyDat$pcp_edu_level[messyDat$pcp_edu_level == 'pg'] <- 'PG'
```

````{warning}
You might be wondering why we did not use `as.numeric()` immediately. R cannot always convert a character string into a number. When it encounters a value it cannot interpret as numeric, it converts that value to `NA` and displays a warning.

```r
myNumbers <- c("1", "2", "three")

as.numeric(myNumbers)
```

```text
[1]  1  2 NA
Warning message:
NAs introduced by coercion
```

Here, `"1"` and `"2"` are successfully converted to numeric values, but `"three"` cannot be converted and becomes `NA`.

In the earlier example, converting `pcp_age` before replacing `"eighteen"` would have turned that value into `NA`. We would then lose the information that the participant's age had been recorded as `"eighteen"`.
````


### Duplicate Rows

It is good practice to check whether a dataset contains duplicate rows. A duplicate row is one where every value is identical to a row that appeared earlier in the data frame.

Use `duplicated()` to identify them:

```r
duplicated(messyDat)
```

```text
[1] FALSE FALSE FALSE FALSE FALSE  TRUE FALSE FALSE FALSE
```

The output contains one logical value for each row:

- `FALSE` means that the row is not a duplicate of an earlier row.
- `TRUE` means that the row duplicates an earlier row.

Here, the sixth row is a duplicate. Whether it should be removed depends on the context. For example, it may be an accidental repeated data entry, or it could represent two different participants whose recorded values happen to be identical. In these data we have a unique participant identifier column so it seems very likely this is an accidental duplication.

If you determine that duplicate rows should be removed, use `!duplicated()` to retain only the first occurrence of each row:

```r
messyDat <- messyDat[!duplicated(messyDat), ]
```

- `duplicated(messyDat)` returns `TRUE` for rows that duplicate an earlier row.
- `!` means “not”, so `!duplicated(messyDat)` returns `TRUE` for rows that should be kept.
- `messyDat[rows, columns]` subsets a data frame. The logical filter before the comma selects rows; leaving nothing after the comma means “keep all columns”.
- `<-` assigns the resulting, de-duplicated data frame back to `messyDat`.

In other words, this code keeps every non-duplicate row, retains all variables, and overwrites `messyDat` with the cleaned version.

## Finding NAs

It is useful to know how many `NA` values there are in your data set and where they are. One way to get a quick overview is to use the `colSums()` function:

```r
colSums(is.na(messyDat))
```

```text
 pcp_id        pcp_age   congruent_rt incongruent_rt  pcp_edu_level  completed_exp 
     1              1              0              1              0              0 
```

This works in two steps:

- `is.na(messyDat)` checks every cell in the data frame and returns `TRUE` where the value is missing (`NA`) and `FALSE` otherwise.
- `colSums()` adds these logical values within each column. In R, `TRUE` is treated as `1` and `FALSE` as `0`, so the result is the number of missing values in each variable.

The output tells us that there is one missing value in the `pcp_id` column, one in the `pcp_age` column and one in the `incongruent_rt` column. What you should do about these is again a matter of judgment. If age has been collected to describe the sample, a missing value is probably not something to worry about. A missing participant identifier is more unusual, so let's have a look at that row:

```r
messyDat[is.na(messyDat$pcp_id),]
```
```text
  pcp_id pcp_age congruent_rt incongruent_rt pcp_edu_level completed_exp
9   <NA>      21         2.89           3.91            UG         FALSE
```
By inspecting this row we can see that the participant did not complete the experiment as shown by the `FALSE` value in the `completed_exp` column. We should probably remove this participant:

```r
messyDat <- messyDat[!is.na(messyDat$pcp_id),]
```


````{warning}
The output above told us that the row with missing `pcp_id` was row 9. So we could have removed that row like this:
```r
messyDat <- messyDat[-9,]
```

This would work for the current version of the dataset. The `-9` means “exclude row 9”, while the comma indicates that all columns should be retained.

However, hard-coding row numbers is usually not good practice. If the data are sorted differently, new rows are added, or another row is removed earlier in the cleaning process, the row with the missing value may no longer be row 9. The code could then remove the wrong participant.

Filtering based on the condition you actually care about is more reliable.
````

## Factors and Categorical Variables

Some variables represent **categories** rather than numerical quantities. For example, a participant's highest level of education might be recorded as:

```text
"school" "UG" "PG" 
```

These values are labels, not measurements. It would not make sense to calculate their mean or add them together.

In the `messyDat` dataset, `pcp_edu_level` contains participants' education levels. It was initially imported as a character variable, but it is better represented as a **factor**.

### Why Use Factors?

A factor is R's data type for categorical variables. It stores a fixed set of possible categories, known as **levels**.

Using a factor helps you to:

- Make it explicit that a variable contains categories.
- Control the order in which categories appear in tables and plots.
- Detect inconsistent category labels, such as `"PG"` and `"pg"`.
- Use an appropriate reference category in statistical models.



### Converting to a Factor

We already standardised the labels earlier, so we can now convert directly using `factor()`

```r
messyDat$pcp_edu_level <- factor(messyDat$pcp_edu_level)
```

Check its type and levels:

```r
class(messyDat$pcp_edu_level)
levels(messyDat$pcp_edu_level)
```

```text
[1] "factor"

[1] "PG"     "School" "UG"    
```

By default, R orders factor levels alphabetically. This affects the order used in tables, plots, and some statistical output.

### Setting a Meaningful Order

Education levels have a natural order, so it is useful to specify it explicitly:

```r
messyDat$pcp_edu_level <- factor(
  messyDat$pcp_edu_level,
  levels = c("School", "UG", "PG"),
  ordered = TRUE
)
```

The `levels` argument defines the categories and their order. Setting `ordered = TRUE` tells R that these categories have a meaningful progression.

Check the result:

```r
levels(messyDat$pcp_edu_level)
```

```text
[1] "School" "UG"     "PG"  
```

### Counting Participants in Each Category

Use `table()` to count participants in each education category:

```r
table(messyDat$pcp_edu_level)
```

Which produces:

```text
School     UG     PG 
     3      2      2 
```

If we had missing values in the table, we could include them with:

```r
table(messyDat$pcp_edu_level, useNA = "ifany")
```

Factors are R's way of representing categorical variables. Before converting a variable to a factor, it is worth checking its unique values and standardising any inconsistent labels. The `levels` argument lets you define both the categories and their order, which matters for tables, plots, and statistical models. When categories have a natural progression, as education levels do, setting `ordered = TRUE` makes that structure explicit.